In [0]:
!pip install beautifulsoup4 requests demjson3 lxml

In [0]:
import requests
from bs4 import BeautifulSoup
from bs4.element import Tag # Import Tag explicitly
from lxml import etree, html # Import html for parsing in lxml
import json
import demjson3
from datetime import datetime, timedelta
import pandas as pd
from zoneinfo import ZoneInfo

In [0]:
arg_time = datetime.now(ZoneInfo("America/Argentina/Buenos_Aires"))
base_volume = "/Volumes/workspace/futbol/futbol_landing"
current_timestamp = (arg_time - timedelta(days=1)).strftime("%Y-%m-%d")
print(f"Timestamp actual: {current_timestamp}")

In [0]:
teams_to_extract = ["Racing", "Boca", "River", "Independiente"]

In [0]:
def scrape_html_content(url, team, html_selector, selector_type=None, multiple=False):
    print(f"URL: {url}")
    """
    Scrapes content from a given URL based on HTML selector and selector type.

    Args:
        url (str): The URL to fetch HTML content from.
        html_selector (str): The selector string (e.g., 'div.content', '//h1', 'p').
        selector_type (str): Type of selector: 'css_selector', 'xpath', 'tag_name', 'class'.
                             Defaults to 'css_selector'.
        multiple (bool): If True, returns a list of all matching elements.
                         If False, returns the first matching element. Defaults to False.
    Returns:
        list or object: A list of scraped contents/elements if 'multiple' is True,
                        or a single content/element if 'multiple' is False.
                        Returns None or empty list on failure or no matches.
    """
    try:
        response = requests.get(url)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    except requests.exceptions.RequestException as e:
        print(f"Error accessing URL {url}: {e}")
        return [] if multiple else None
    html_content = response.text
    found_elements = []
    if selector_type == 'xpath':
        # Use lxml for both parsing and selection if selector_type is xpath
        try:
            tree = html.fromstring(html_content)
            found_elements = tree.xpath(html_selector)
            # Removed: print(f"LIST OF ELEMENTS FOUNDED: {found_elements} BY XPATH")
        except Exception as e:
            print(f"Error parsing with lxml or executing XPath: {e}")
            return [] if multiple else None
    else:
        # Use BeautifulSoup for other selector types
        soup = BeautifulSoup(html_content, 'html.parser')
        if selector_type == 'css_selector':
            found_elements = soup.select(html_selector)
        elif selector_type == 'tag_name':
            found_elements = soup.find_all(html_selector)
        elif selector_type == 'class':
            # html_selector should be just the class name here
            found_elements = soup.find_all(class_=html_selector)
        else:
            print(f"Unsupported selector_type: {selector_type} for BeautifulSoup.")
            return [] if multiple else None
    parsed_elements = []
    for element in found_elements:
        if element is None or element.text is None:
            continue
        try:
            extracted_content = json.loads(element.text)
            if (
                "/fecha/" in url
                and
                extracted_content.get("homeTeam", {}).get("name") == team
                or
                extracted_content.get("awayTeam", {}).get("name") == team
            ):
                print(f"Element found: {extracted_content}")
                parsed_elements.append(extracted_content)
            elif "/futbol" in url:
                print(f"Element found in match url: {extracted_content}")
                parsed_elements.append(extracted_content)
        except Exception:
            pass
    print(f"ELEMENTS FOUND ---> {parsed_elements}")
    return parsed_elements

In [0]:
def run(date, team):
    matchs_scripts = scrape_html_content(
        f"https://canchallena.lanacion.com.ar/fecha/{date}/",
        team,
        '//script[@type="application/ld+json" and @data-react-helmet="true"]',
        selector_type="xpath",
        multiple=True
    )
    current_matches = []
    for match in matchs_scripts:
        if match.get("@type", "") == "SportsEvent":
            name = match.get("name", "")
            link = match.get("url", "")
            print(f"LINK: {link}")
            start_date = match.get("startDate", "")
            league = match.get("organizer", {}).get("name", "")
            if link is not None:
                match_details = scrape_html_content(
                    link,
                    team,
                    '//script[@type="application/ld+json" and @data-react-helmet="true"]',
                    selector_type="xpath",
                    multiple=True
                )
                current_match = {
                    "scrape_date": date,
                    "date": start_date,
                    "name": name,
                    "league": league,
                    "link": link,
                    "match": match_details
                }
                current_matches.append(current_match)
    year, month, day = date.split("-")
    path = f"{base_volume}/raw/year={year}/month={month}/day={day}"
    df_current_match = pd.json_normalize(current_matches, max_level=1)
    if not df_current_match.empty and "match" in df_current_match.columns:
        df_current_match = df_current_match[
            df_current_match["match"].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)
        ]
    print(f"DF CURRENT MATCH -----> \n{df_current_match.head()} -----> \n{df_current_match.info()}")
    if not df_current_match.empty:
        dbutils.fs.mkdirs(path)
        full_path = f"{path}/{team}_{date}.parquet"
        df_current_match.to_parquet(full_path, index=False)
        print(f"✅ Guardado: {full_path}")
        print("")
        print("")
        print("")

In [0]:
for team in teams_to_extract:
    run(current_timestamp, team)

In [0]:
# from datetime import datetime, timedelta

# start_date = datetime(2023, 1, 1)
# # end_date = datetime(2024, 12, 31)
# end_date = datetime.now() - timedelta(days=1)

# date_range = []
# current = start_date
# while current <= end_date:
#     date_range.append(current.strftime("%Y-%m-%d")[0:10])
#     current += timedelta(days=1)

# for team in teams_to_extract:
#     for date_str in date_range:
#         print(date_str)
#         run(date_str, team)